# 🎨 Notebook 1: Airline Management — Class Design

We'll design an airline booking system the way a real engineer does it:

1. **Gather requirements** in plain English.
2. **Try a naive design** and see where it hurts.
3. **Refactor** toward a clean, extensible model.

By the end, you'll understand *why* each class exists — not just *what* it does.


## 🛠️ Setup

```bash
cd 07-object-oriented-design/airline-management
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 📋 Requirements (what the product must do)

- Airline operates many **flights**. A flight has an origin, destination, departure time, and an **aircraft** assigned to it.
- An **aircraft** has a fixed seat map (e.g., A320 with 150 seats). The *same plane* flies *many flights* over its life.
- Passengers can **book a seat** on a specific flight. A seat on a given flight can only be booked once.
- Seats come in **classes**: Economy, Business, First. Price depends on class.
- Passengers can **cancel** (refund depends on how early they cancel).
- Users can **search** flights by origin, destination, and date.
- Flights have **crew** (pilots, attendants).

Let's start naively and refactor.


## 🚫 Attempt 1 (bad): one giant `Flight` class that does everything

A common beginner mistake is to cram *all* the logic into one class — seats, prices, bookings, crew — because "it's all about the flight, right?"


In [ ]:
# BAD: a God class. Works for a demo, falls apart in real life.
class BadFlight:
    def __init__(self, number, origin, dest, total_seats):
        self.number = number
        self.origin = origin
        self.dest = dest
        # seats are just strings; class info is encoded in the seat name 🤢
        self.seats = [f"E{i}" for i in range(total_seats)]
        self.booked = set()
        self.passengers = {}  # seat -> passenger name
        self.crew = []        # free-form strings

    def book(self, seat, passenger):
        if seat in self.booked:
            raise ValueError("taken")
        # Price is hard-coded here. What if we add a promo? Edit this method.
        price = 500 if seat.startswith("E") else 1500
        self.booked.add(seat)
        self.passengers[seat] = passenger
        return price

bf = BadFlight("UA1", "SFO", "JFK", 3)
print(bf.book("E0", "Alice"), bf.book("E1", "Bob"))
print("state:", bf.passengers)


### Why this design hurts

- **No reuse.** The same A320 aircraft is used by 20 flights a week, but here every `BadFlight` re-invents its own seat list.
- **Seat *class* is encoded in a string prefix.** Adding a new class (e.g., Premium Economy) means scanning every method that parses seat names.
- **Price is hard-coded inside `book`.** Any promo, tax, or loyalty discount forces editing this method → violates the **Open/Closed Principle**.
- **Crew, passengers, bookings, pricing, seat map — all in one class.** Violates **Single Responsibility**. Hard to test, hard to change.


## 🚫 Attempt 2 (still bad): a subclass per seat class

"OK, let's use OOP properly — I'll make a subclass for each seat class!"


In [ ]:
# BAD: subclass explosion. Each new class/promo/region doubles the types.
class Seat:
    def __init__(self, number):
        self.number = number
    def price(self):
        raise NotImplementedError

class EconomySeat(Seat):
    def price(self): return 200

class BusinessSeat(Seat):
    def price(self): return 700

class FirstSeat(Seat):
    def price(self): return 1800

# Now imagine Premium Economy, plus a Holiday promo, plus a Student discount…
# You get EconomyStudentHolidaySeat. This is the "subclass explosion" anti-pattern.
print(EconomySeat("12A").price(), FirstSeat("1A").price())


### Why this still hurts

- **Class is data, not behavior.** Economy and Business seats do the same thing — they have a price. Different *value*, not different *behavior*. Using subclasses here is "nouns gone wild."
- **Adding a new fare class = new subclass everywhere.** Can't just add a row to a pricing table.
- **Hard to combine variations** (promo × class × loyalty) — the class tree explodes.

**Rule of thumb:** subclass when *behavior* differs, not when *data* differs. Here, use an `enum` + a lookup table.


## ✅ Best design: separate concerns, use enums for data-shaped variation

```
   Aircraft ──1──◆── Seat         (the physical plane + its seat map)
       ▲
       │ assigned to
   Flight  ──1──◆── per-flight seat availability
       │
       │ 1..*
       ▼
   Booking ──1──▶ Ticket ──▶ Seat
       │
       │ for
       ▼
   Passenger

   Flight ──*──── Crew            (pilots + attendants)
   FlightSearch ─ queries many Flights
```

### Key decisions and *why*

| Decision | Reason |
|---|---|
| `Aircraft` owns the **seat map** | Same plane → many flights. Don't duplicate seats. |
| `Flight` owns **availability** (seat → booked?) | Availability is per-flight, not per-plane. |
| `SeatClass` is an **enum**, price is a **table** | Class varies by *data*, not *behavior*. Add a class = add a row. |
| `Booking` is its own class | Encapsulates passenger + flight + seat + price + status. Easy to cancel, audit, refund. |
| `FlightSearch` is a **separate service** | Search is a query across many flights — doesn't belong on `Flight`. |
| `RefundPolicy` is a **Strategy** | The refund rule changes per fare and per promo. Behaviour varies → inject an object, don't edit `cancel`. |
| `Crew` is a separate concept from `Passenger` | Different lifecycle, different behavior. |

### Classes at a glance

| Class | Role |
|---|---|
| `Passenger` | id, name, passport |
| `SeatClass` | enum: ECONOMY / BUSINESS / FIRST |
| `Seat` | number (e.g., `12A`), `seat_class` |
| `Aircraft` | model + list of `Seat` |
| `Flight` | number, origin, dest, departs, aircraft, availability map |
| `Booking` | passenger + flight + seat + price + status |
| `CrewMember` | id, name, role (pilot / attendant) |
| `FlightSearch` | finds flights by origin / dest / date |
| `RefundPolicy` | **Strategy**: how much money a cancellation returns |

Notebook 2 implements all of this and exercises it end-to-end.
